# Retrieval Experiments

Compare TF-IDF, BM25, Dense, Hybrid retrieval; chunking strategies; and embedding models on Amazon Fine Food Reviews.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px

from src.config.settings import get_settings
from src.evaluation.experiment_runner import evaluate_retriever
from src.embeddings.chunking import build_chunk_corpus
from src.utils.io import load_parquet

settings = get_settings()
df = load_parquet(settings.processed)
print(f"Loaded {len(df):,} reviews")

## Experiment 1: TF-IDF vs BM25

In [ ]:
bench_tfidf, _ = evaluate_retriever(df, "tfidf")
bench_bm25, _ = evaluate_retriever(df, "bm25")
exp1 = pd.concat([bench_tfidf, bench_bm25], ignore_index=True)
exp1_agg = exp1.groupby("retriever")[[c for c in exp1.columns if "precision" in c or c == "mrr"]].mean().reset_index()
px.bar(exp1_agg, x="retriever", y=exp1_agg.columns[1], title="Experiment 1: TF-IDF vs BM25").show()

## Experiment 2: BM25 vs Dense (mpnet)

In [ ]:
bench_dense, _ = evaluate_retriever(df, "dense", model_name="all-mpnet-base-v2")
exp2 = pd.concat([bench_bm25, bench_dense], ignore_index=True)
exp2_agg = exp2.groupby("retriever")["mrr", "latency_ms"].mean().reset_index()
px.scatter(exp2_agg, x="latency_ms", y="mrr", color="retriever", title="Experiment 2: BM25 vs Dense").show()

## Experiment 3: Dense vs Hybrid

In [ ]:
bench_hybrid, _ = evaluate_retriever(df, "hybrid")
exp3 = pd.concat([bench_dense, bench_hybrid], ignore_index=True)
prec_col = [c for c in exp3.columns if "precision" in c][0]
px.bar(exp3.groupby("retriever")[prec_col].mean().reset_index(), x="retriever", y=prec_col, title="Experiment 3: Dense vs Hybrid").show()

## Experiment 4: Chunking Strategies

In [ ]:
chunk_stats = []
for strategy in ["fixed", "overlap", "semantic"]:
    chunks = build_chunk_corpus(df.head(1000), strategy=strategy)
    chunk_stats.append({"strategy": strategy, "chunk_count": len(chunks), "avg_chunks_per_review": len(chunks)/1000})
chunk_df = pd.DataFrame(chunk_stats)
px.bar(chunk_df, x="strategy", y="chunk_count", title="Experiment 4: Chunk Count by Strategy").show()

## Experiment 5: mpnet vs bge Embeddings

In [ ]:
bench_bge, _ = evaluate_retriever(df, "dense", model_name="BAAI/bge-small-en-v1.5")
exp5 = pd.concat([bench_dense.assign(model="mpnet"), bench_bge.assign(model="bge")], ignore_index=True)
px.box(exp5, x="model", y="mrr", title="Experiment 5: Embedding Model Comparison").show()

all_bench = pd.concat([exp1, bench_dense, bench_hybrid, exp5], ignore_index=True)
all_bench.to_csv(settings.reports / "retrieval_benchmark_notebook.csv", index=False)
print("Saved notebook benchmark to reports/retrieval_benchmark_notebook.csv")